# 05 — Geometry on Real Hardware: the Bures Distance

How far did the device's state fall from the ideal you asked for? In `05/03` you answered with a fidelity below 1 — an honest score, but a *number*, not yet a *distance*. Here you turn it into one. The gap becomes a single point on the Bures geometry you built in `02/13`, and then you add circuit depth and test whether that point moves outward. On the offline model of the device it does, step by step; on the real device, one pass per depth could not resolve the rise.

**Prerequisites:** `02/13_bures_distance`, `05/03_tomography_and_fidelity_on_real_hardware`, `05/04_quantum_mutual_information_on_real_hardware`.

**What you'll be able to do**
- Reconstruct a device state exactly as in `05/03` and measure `bures_distance(rho_hat, ideal)` — the decoherence gap read off the Bures geometry as a single number.
- Dial decoherence up with identity padding and measure, with error bars, how the Bures distance and the fidelity respond: two views of one physical effect.
- Place the device on the very geometry that runs through modules 02–04, the geometry that powers quantum optimal transport.

You already have every piece. `05/03` reconstructed `rho_hat` from a device's clicks and scored its fidelity; `02/13` built the Bures distance as the quantum analogue of the Fisher–Rao metric. Here those two threads meet: you feed the device's reconstructed state straight into the Bures distance and read the decoherence gap as a geometric length. Then, by lengthening the circuit, you test whether that length grows, with error bars that tell a trend from shot noise. Let us go and measure it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2
from qot_course.hardware.records import save_run_record
from qot_course.hardware.runtime import select_backend
from qot_course.hardware.tomography import single_qubit_tomography_circuits, density_from_counts
from qot_course.infotheory.quantum import bures_distance
from qot_course.quantum.density import density_matrix, fidelity
from qot_course.quantum.states import KET_PLUS

np.random.seed(42)
viz.use_course_style()

In [ ]:
USE_HARDWARE = False  # flip to True after you've set up credentials (see 05/01)
backend, label, is_real = select_backend(use_hardware=USE_HARDWARE)
print(f"Running on: {label}  (real hardware = {is_real})")
if not is_real:
    backend.set_options(seed_simulator=42)  # reproducible offline figure; a real QPU has no such option

## 1 · A geometric read-out of decoherence

In `05/03` the fidelity told you *how close* the device's state landed to the ideal, on a scale from 0 to 1. The Bures distance turns that closeness into a genuine length. For two states $\hat\rho$ and the ideal target it is

$$d_B(\hat\rho, \mathrm{ideal}) = \sqrt{2\,\bigl(1 - \sqrt{F}\bigr)},$$

where $F$ is the fidelity of `01/14` in the course's squared convention ($F = \langle\psi|\hat\rho|\psi\rangle$ for a pure target), so that $\sqrt F$ is the Uhlmann fidelity $F_U$; `bures_distance` takes that root for you. This is the Bures distance you built in `02/13`. It is a chord, the straight-line separation of the two states, and not the length of the geodesic arc of the Bures geometry; like the arc, it is 0 exactly when the states are identical and grows as they separate. It is the quantum sibling of the classical Fisher–Rao distance from `02/07` — the same information-geometric idea, lifted from probability vectors to density matrices.

The plan is direct. We reconstruct $\hat\rho$ exactly as in `05/03` — single-qubit Pauli tomography through `SamplerV2`, reassembled by `density_from_counts` — and hand it to `bures_distance`. The decoherence gap that `05/03` reported as a fidelity now reads as a *distance*: one number locating the device on the geometry.

## 2 · Dialing decoherence

A single reading is one point. To watch the geometry *respond*, we need a knob that adds error in controlled amounts — and the cleanest knob is circuit depth that leaves the target untouched.

We insert $k$ identity layers before tomography. Each layer is an `x` followed by another `x`: $X \cdot X = I$, so the state we *intend* to prepare never changes — it stays $|+\rangle$ for every $k$. What does change is the number of physical gates the device must execute, and every real gate leaks a little error. More depth means more accumulated gate error, so the reconstructed $\hat\rho$ should drift further from the ideal, and the Bures distance with it. Each depth is run eight times, so that its error bar can be read off the spread of the eight values.

There is one subtlety that makes or breaks the experiment. A transpiler is built to *simplify* circuits, and an $X \cdot X$ pair is exactly the kind of redundancy it loves to cancel — left alone, it would erase every padding gate and the sweep would go flat. To stop it, we wrap each `x` in barriers: `prep.barrier(); prep.x(0); prep.barrier(); prep.x(0)`. A barrier forbids the transpiler from moving gates across it, so the pairs survive transpilation and actually run on the device. Without the barriers there is no decoherence to measure; with them, the error accumulates as intended.

In [ ]:
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
ideal = density_matrix(KET_PLUS)
k_values = [0, 4, 8, 16]
N_REP = 8          # independent repeats of each depth; their spread is the error bar of one run
SHOTS = 8192       # per setting; on a real QPU this sweep is one job of 4 x 8 x 3 = 96 circuits
pubs, keys = [], []
for k in k_values:
    prep = QuantumCircuit(1)
    prep.h(0)
    for _ in range(k):                       # identity padding (X X = I); barriers stop the transpiler cancelling it
        prep.barrier(); prep.x(0); prep.barrier(); prep.x(0)
    circuits = single_qubit_tomography_circuits(prep)
    isa = {s: pm.run(circuits[s]) for s in circuits}
    for rep in range(N_REP):
        for s in ("X", "Y", "Z"):
            pubs.append(isa[s]); keys.append((k, rep, s))
job = SamplerV2(mode=backend).run(pubs, shots=SHOTS)
res = job.result()
all_counts = {f"k{k:02d}_rep{rep}_{s}": res[i].data.c.get_counts() for i, (k, rep, s) in enumerate(keys)}
save_run_record(all_counts, notebook="05_05", tag="depth_sweep", device=backend.name, shots=SHOTS,
                job_id=job.job_id() if is_real else None, is_real=is_real,
                extra={"k_values": k_values, "n_rep": N_REP})

bures = np.zeros((len(k_values), N_REP))
fids = np.zeros((len(k_values), N_REP))
for a, k in enumerate(k_values):
    for rep in range(N_REP):
        cbs = {s: all_counts[f"k{k:02d}_rep{rep}_{s}"] for s in ("X", "Y", "Z")}
        rho = density_from_counts(cbs, n_qubits=1)
        bures[a, rep] = bures_distance(rho, ideal)
        fids[a, rep] = fidelity(rho, ideal)
bures_mean, bures_sd = bures.mean(axis=1), bures.std(axis=1, ddof=1)
fids_mean, fids_sd = fids.mean(axis=1), fids.std(axis=1, ddof=1)
print(f"{N_REP} repeats per depth, {SHOTS} shots per setting; mean +/- spread of one run (s.e. of the mean = spread / {np.sqrt(N_REP):.2f})")
for k, d, dsd, f, fsd in zip(k_values, bures_mean, bures_sd, fids_mean, fids_sd):
    print(f"padding k={k:2d} ({2 * k:2d} X gates): Bures = {d:.4f} +/- {dsd:.4f}, fidelity = {f:.4f} +/- {fsd:.4f}")
rise = bures_mean[-1] - bures_mean[0]
print(f"rise of the mean Bures distance from k=0 to k=16: {rise:.4f} "
      f"= {rise / bures_sd[0]:.1f} times the spread of one run at k=0 ({bures_sd[0]:.4f})")
print("mean Bures distance increases at every step:", bool(np.all(np.diff(bures_mean) > 0)))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.errorbar(k_values, bures_mean, yerr=bures_sd, fmt="o-", color=COLORS["quantum"], capsize=4)
ax1.set_xlabel("identity-padding depth k"); ax1.set_ylabel("Bures distance to |+>")
ax1.set_title("Bures distance against depth (mean +/- spread of one run)", pad=10)
ax2.errorbar(k_values, fids_mean, yerr=fids_sd, fmt="s-", color=COLORS["flow"], capsize=4)
ax2.set_xlabel("identity-padding depth k"); ax2.set_ylabel("fidelity to |+>")
ax2.set_title("...and fidelity against depth", pad=10)
fig.tight_layout()
plt.show()

**Read the figure.** Two views of one physical effect. Each point is the mean of eight runs, and its error bar is the spread of one run. On the offline model the mean Bures distance rises at every step, from $0.1044$ at $k=0$ to $0.1421$ at $k=16$, while the fidelity falls from $0.9891$ to $0.9799$. The rise, $0.0377$, is about nine times the spread of one run at $k=0$, so it is resolved and not shot noise. The $k=0$ baseline is not zero, and that honesty matters: it is the Hadamard-prep error plus the read-out error, together with finite-shot and projection bias from reconstructing $\hat\rho$ from a finite number of clicks. The *rise* across the sweep is the part attributable to the added `x` gates, each padding layer two more chances to err. (Offline this reflects the modeled FakeMarrakesh noise; flip `USE_HARDWARE=True` and the very same curves report the literal chip.)

### A real run on IBM hardware

On **`ibm_marrakesh`** (2026-05-30), the same depth sweep, one pass per depth, gave Bures distances of about **0.116, 0.127, 0.136, 0.125** for k = 0, 4, 8, 16 (fidelity about 0.98 throughout). The job identifier and the raw counts of that run were not kept, so these values cannot be recomputed from this repository. The offline twin of the same device puts the spread of one run between $0.004$ and $0.006$ (above), so the real points moved by more than shot noise, and not monotonically: the $k=16$ point fell back below $k=8$. One unrepeated pass per depth cannot say why, whether the device drifted between circuits or carries noise the model leaves out, and it does not resolve the steady rise the offline twin shows. The repeated sweep above is the test that would: set `USE_HARDWARE = True` to run it, as one job of 96 circuits.

## 3 · The geometry behind this distance

Step back and notice what this distance *is*. The Bures geometry you measured a device on is the same one you built in `02/13`, the same one bridged to the Wasserstein distance of optimal transport in `03/12–13`, and the same one underlying the quantum optimal-transport measures of module 04. The geometry that powers quantum optimal transport is exactly the geometry that quantified a real chip's error a moment ago. The device's decoherence and the mathematics of QOT are not two subjects — they are one geometry, met from two directions.

## Your turn

- **Swap the padding for a `delay`.** If your backend supports the `delay` instruction, replace the `x·x` layers with an idle `delay` of growing duration and compare the curves. Gate error and idle decoherence (T1/T2) are different physical channels — does pure idling move the Bures distance the same way the gates do?
- **Flip `USE_HARDWARE = True`** (with credentials set per `05/01`) and re-run. The distances you read are now the real chip's drift along the Bures geometry, not a model's.
- **Check the closed form.** Take one run, say `fids[0, 0]` and `bures[0, 0]`, compute `np.sqrt(2 * (1 - np.sqrt(fids[0, 0])))` and compare it with `bures[0, 0]`: the same number, once read off the device and once off the formula of `02/13`. Then try the same check on the printed *means*. Why do they agree only approximately?

## What you built

- A geometric read-out of decoherence: you reconstructed a device state with single-qubit tomography and measured `bures_distance(rho_hat, ideal)`, turning the fidelity gap of `05/03` into a genuine distance on the Bures geometry.
- A controlled decoherence sweep: identity padding (with barriers to survive transpilation) that lengthens the circuit while leaving the target fixed, so you could measure, with error bars, the Bures distance rise and the fidelity fall in step on the offline model (the real run of 2026-05-30 did not resolve the rise).
- An honest decomposition of the gap: a nonzero k=0 baseline (prep + read-out + finite-shot/projection bias) and a clean *rise* attributable to the added gate error.
- The closing of the module's geometric arc: the device placed on the same Bures geometry that runs through modules 02–04 and powers quantum optimal transport.

## What's next

Next is the **capstone** of the real-hardware module: you turn the mutual-information measurement of `05/04` into an instrument, measure how finely a device resolves it, and set that resolution beside the gaps of the Module 04 capstone.

## References

- D. Bures, "An extension of Kakutani's theorem on infinite product measures to the tensor product of semifinite $w^*$-algebras," *Trans. Amer. Math. Soc.* **135**, 199 (1969).
- M. M. Wilde, *Quantum Information Theory*, ch. 9, Cambridge University Press (2017), doi:10.1017/9781316809976.
- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 9 (fidelity and distance measures), Cambridge University Press (2010).

**Previous:** `notebooks/05_RealQuantumHardware/04_quantum_mutual_information_on_real_hardware.ipynb` · **Next:** `notebooks/05_RealQuantumHardware/06_capstone_on_real_hardware_and_frontier.ipynb`